# 🧬 Biyolojide Yapay Zeka (Bio-AI) - Vize Projesi
# Likit Biyopside Hücre Dışı DNA (cfDNA) Fragmantasyon Profilleri (DELFI) ile Erken Evre Kanser Tespiti

**Öğrenci:** Sude Merve Bulut  
**Öğrenci No:** 03190321083  
**Ders:** Biyolojide Yapay Zeka (*Artificial Intelligence in Biology*)  
**Alan:** Kanser Genomikleri, Epigenetik & Likit Biyopsi Biyoinformatiği  
**Programlama Dili:** Python 3  

---

## 📌 1. Biyolojik Problem Tanımı ve Klinik Önemi
Kanser tedavisinde sağkalımı artıran en kritik faktör hastalığın **Evre I veya Evre II** aşamasında, henüz metastaz gerçekleşmeden tespit edilmesidir. Geleneksel doku biyopsileri invazivdir ve her organdan kolayca alınamaz. **Likit biyopsi**, basit bir kan örneğindeki plazmada serbest dolaşan **hücre dışı DNA (cell-free DNA - cfDNA)** moleküllerini analiz ederek bu sorunu çözer.

### 🔬 Fragmanomik (Fragmentomics) Yaklaşımı Nedir?
1. **Sağlıklı Hücrelerde Nükleozom Düzeni:** Sağlıklı bireylerde apoptoz yoluyla kana karışan cfDNA molekülleri histon proteinlerine sarılı olduğundan enzimatik kesime dirençlidir ve **~167 baz çifti (bp)** uzunluğunda keskin bir tepe noktası oluşturur.
2. **Kanser Hücrelerinde Kromatin Düzensizliği:** Tümör hücrelerinden kaynaklanan dolaşımdaki tümör DNA'sı (**ctDNA**), açık kromatin yapısı ve değişmiş nükleozom paketlenmesi nedeniyle daha küçük parçalara (**100–150 bp**) ayrılır.
3. **DELFI Metodolojisi:** Genom boyunca 39 otozomal kromozom kolundaki **Kısa/Uzun Fragman Oranları (Short-to-Long Ratio)** ve 5' uç motifleri (**CCCA end-motif**) makine öğrenmesi ile analiz edilerek hem kanser varlığı hem de **tümörün doku kökeni (Akciğer vs. Kolorektal)** tespit edilir.

## 2. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

print("✅ Kütüphaneler başarıyla yüklendi!")

## 3. Likit Biyopsi cfDNA Fragmanomik Veri Seti Simülasyonu
39 kromozom kolu için Short (100-150 bp) / Long (151-220 bp) oranları, plazma cfDNA konsantrasyonu (ng/mL), modal fragman uzunluğu (bp) ve 5' uç motif frekansı modellenir:
- **0 - Sağlıklı Kontrol:** Stabil SL oranı (~0.18), 167 bp modal boyut.
- **1 - Erken Evre Akciğer Kanseri (NSCLC):** 1q, 3q, 5p, 7p, 8q, 17p, 19q kollarında fragman kısalması.
- **2 - Erken Evre Kolorektal Kanser (CRC):** 7p, 8q, 13q, 17p, 18q, 20p, 20q kollarında fragman düzensizliği.

In [ ]:
CHROMOSOME_ARMS = [
    "1p", "1q", "2p", "2q", "3p", "3q", "4p", "4q", "5p", "5q",
    "6p", "6q", "7p", "7q", "8p", "8q", "9p", "9q", "10p", "10q",
    "11p", "11q", "12p", "12q", "13q", "14q", "15q", "16p", "16q",
    "17p", "17q", "18p", "18q", "19p", "19q", "20p", "20q", "21q", "22q"
]

CLASS_NAMES = ["Sağlıklı Kontrol", "Akciğer Kanseri (NSCLC)", "Kolorektal Kanser (CRC)"]

def generate_cfdna_dataset(n_samples: int = 720, random_seed: int = 104) -> pd.DataFrame:
    np.random.seed(random_seed)
    data = []
    n_healthy = int(n_samples * 0.45)
    n_nsclc = int(n_samples * 0.30)
    n_crc = n_samples - n_healthy - n_nsclc
    sample_id = 1

    for _ in range(n_healthy):
        row = {
            "sample_id": f"LIQ_{sample_id:04d}",
            "patient_age": int(np.clip(np.random.normal(54, 9.5), 30, 80)),
            "cfdna_ng_ml": round(np.clip(np.random.normal(8.5, 2.2), 3.0, 16.0), 2),
            "modal_fragment_bp": int(np.random.normal(167, 1.5)),
            "motif_CCCA_freq": round(np.random.normal(0.024, 0.002), 5),
            "target_class": 0
        }
        base = np.random.normal(0.18, 0.015, size=len(CHROMOSOME_ARMS))
        for idx, arm in enumerate(CHROMOSOME_ARMS):
            row[f"sl_ratio_{arm}"] = round(max(0.08, base[idx]), 4)
        data.append(row)
        sample_id += 1

    nsclc_hotspots = ["1q", "3q", "5p", "7p", "8q", "17p", "19q"]
    for _ in range(n_nsclc):
        row = {
            "sample_id": f"LIQ_{sample_id:04d}",
            "patient_age": int(np.clip(np.random.normal(63, 8.0), 40, 85)),
            "cfdna_ng_ml": round(np.clip(np.random.normal(11.8, 3.8), 6.0, 35.0), 2),
            "modal_fragment_bp": int(np.random.normal(164.5, 2.6)),
            "motif_CCCA_freq": round(np.random.normal(0.020, 0.0028), 5),
            "target_class": 1
        }
        base = np.random.normal(0.205, 0.038, size=len(CHROMOSOME_ARMS))
        for idx, arm in enumerate(CHROMOSOME_ARMS):
            val = base[idx] + (np.random.normal(0.068, 0.030) if arm in nsclc_hotspots else 0)
            row[f"sl_ratio_{arm}"] = round(max(0.10, val), 4)
        data.append(row)
        sample_id += 1

    crc_hotspots = ["7p", "8q", "13q", "17p", "18q", "20p", "20q"]
    for _ in range(n_crc):
        row = {
            "sample_id": f"LIQ_{sample_id:04d}",
            "patient_age": int(np.clip(np.random.normal(61, 8.5), 38, 84)),
            "cfdna_ng_ml": round(np.clip(np.random.normal(12.4, 4.1), 6.5, 38.0), 2),
            "modal_fragment_bp": int(np.random.normal(164.0, 2.8)),
            "motif_CCCA_freq": round(np.random.normal(0.0195, 0.0028), 5),
            "target_class": 2
        }
        base = np.random.normal(0.210, 0.040, size=len(CHROMOSOME_ARMS))
        for idx, arm in enumerate(CHROMOSOME_ARMS):
            val = base[idx] + (np.random.normal(0.072, 0.032) if arm in crc_hotspots else 0)
            row[f"sl_ratio_{arm}"] = round(max(0.10, val), 4)
        data.append(row)
        sample_id += 1

    return pd.DataFrame(data).sample(frac=1.0, random_state=random_seed).reset_index(drop=True)

raw_df = generate_cfdna_dataset(720)
print(f"Simüle Edilen Plazma Örneği: {raw_df.shape[0]}")
raw_df.head()

## 4. Fragmanomik Biyoinformatik Öznitelik Mühendisliği
- **Shannon Fragmantasyon Entropisi:** Kromatin heterojenitesini ölçer.
- **Sağlıklı Referans Kosinüs Benzerliği:** Hastanın genomik profilinin sağlıklı referanstan sapma derecesini hesaplar.
- **Doku Kökeni İndeksleri:** Akciğer ve Kolorektal kanser için spesifik kromozom kolu sapma skorları.

In [ ]:
def extract_fragmentomic_features(df: pd.DataFrame) -> pd.DataFrame:
    df_feat = df.copy()
    arm_cols = [f"sl_ratio_{arm}" for arm in CHROMOSOME_ARMS]
    sl_matrix = df_feat[arm_cols].values

    df_feat["global_mean_sl"] = np.mean(sl_matrix, axis=1)
    df_feat["global_std_sl"] = np.std(sl_matrix, axis=1)

    prob_matrix = sl_matrix / np.sum(sl_matrix, axis=1, keepdims=True)
    df_feat["fragment_shannon_entropy"] = entropy(prob_matrix, axis=1)

    healthy_ref = np.full(len(CHROMOSOME_ARMS), 0.18)
    norms = np.linalg.norm(sl_matrix, axis=1) * np.linalg.norm(healthy_ref)
    df_feat["healthy_cosine_sim"] = np.dot(sl_matrix, healthy_ref) / norms

    nsclc_cols = [f"sl_ratio_{arm}" for arm in ["1q", "3q", "5p", "19q"]]
    crc_cols = [f"sl_ratio_{arm}" for arm in ["13q", "18q", "20p", "20q"]]

    df_feat["nsclc_signature_index"] = df_feat[nsclc_cols].mean(axis=1) - df_feat["global_mean_sl"]
    df_feat["crc_signature_index"] = df_feat[crc_cols].mean(axis=1) - df_feat["global_mean_sl"]
    df_feat["aberrant_arms_count"] = np.sum(sl_matrix > 0.25, axis=1)
    return df_feat

df_featured = extract_fragmentomic_features(raw_df)
feature_cols = [c for c in df_featured.columns if c not in ["sample_id", "target_class"]]
print(f"Toplam Çıkarılan Özellik Sayısı: {len(feature_cols)}")

## 5. Çoklu Makine Öğrenmesi Modellerinin Eğitimi ve 5-Katlı Çapraz Doğrulama

In [ ]:
X = df_featured[feature_cols].values
y = df_featured["target_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=104, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Extra Trees Classifier": ExtraTreesClassifier(n_estimators=200, max_depth=12, random_state=104),
    "Gradient Boosting Classifier": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, random_state=104),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=104),
    "Logistic Regression (L2)": LogisticRegression(max_iter=1000, C=1.5, random_state=104)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=104)
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1_macro')
    print(f"• {name:<30}: 5-Fold Macro F1 = {scores.mean():.4f} (± {scores.std():.4f})")

## 6. Bağımsız Test Kümesi Değerlendirmesi ve Metrik Karşılaştırması

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": precision_score(y_test, y_pred, average='macro'),
        "Macro Recall": recall_score(y_test, y_pred, average='macro'),
        "Macro F1": f1_score(y_test, y_pred, average='macro'),
        "Macro ROC-AUC": roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    })

metrics_df = pd.DataFrame(results).sort_values(by="Macro ROC-AUC", ascending=False)
display(metrics_df)

best_model_name = metrics_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"\n★ En İyi Model: {best_model_name}")
print(classification_report(y_test, best_model.predict(X_test_scaled), target_names=CLASS_NAMES))

## 7. Görselleştirme: Karışıklık Matrisi, ROC Eğrileri, Biyobelirteçler ve DELFI Profili

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
plt.subplots_adjust(wspace=0.25, hspace=0.35)

# 1. Karışıklık Matrisi
cm = confusion_matrix(y_test, best_model.predict(X_test_scaled))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0, 0],
            xticklabels=["Sağlıklı", "Akciğer", "Kolorektal"],
            yticklabels=["Sağlıklı", "Akciğer", "Kolorektal"])
axes[0, 0].set_title(f"Karışıklık Matrisi ({best_model_name})", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Yapay Zeka Tahmini")
axes[0, 0].set_ylabel("Gerçek Klinik Durum")

# 2. ROC-AUC Eğrileri
y_prob_best = best_model.predict_proba(X_test_scaled)
colors = ['#2ca02c', '#1f77b4', '#d62728']
for idx, (label, col) in enumerate(zip(CLASS_NAMES, colors)):
    y_bin = (y_test == idx).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_prob_best[:, idx])
    auc_score = roc_auc_score(y_bin, y_prob_best[:, idx])
    axes[0, 1].plot(fpr, tpr, lw=2.2, color=col, label=f"{label} (AUC = {auc_score:.3f})")

axes[0, 1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0, 1].set_title("Sınıf Bazında ROC-AUC Eğrileri", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].legend(loc="lower right")
axes[0, 1].grid(alpha=0.3)

# 3. Biyobelirteç Önemi
importances = trained_models["Random Forest Classifier"].feature_importances_
top_idx = np.argsort(importances)[-12:]
axes[1, 0].barh(range(len(top_idx)), importances[top_idx], color='#008080')
axes[1, 0].set_yticks(range(len(top_idx)))
axes[1, 0].set_yticklabels([feature_cols[i] for i in top_idx])
axes[1, 0].set_title("En Belirleyici 12 cfDNA Fragmanomik Biyobelirteç", fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. 39 Kromozom Kolu DELFI Profili
s_healthy = raw_df[raw_df["target_class"] == 0].iloc[0]
s_nsclc = raw_df[raw_df["target_class"] == 1].iloc[0]
s_crc = raw_df[raw_df["target_class"] == 2].iloc[0]
x_arms = np.arange(len(CHROMOSOME_ARMS))

axes[1, 1].plot(x_arms, [s_healthy[f"sl_ratio_{a}"] for a in CHROMOSOME_ARMS], 'o-', color='#2ca02c', label="Sağlıklı Kontrol", lw=1.5, markersize=4)
axes[1, 1].plot(x_arms, [s_nsclc[f"sl_ratio_{a}"] for a in CHROMOSOME_ARMS], 's-', color='#1f77b4', label="Akciğer Kanseri (NSCLC)", lw=1.5, markersize=4)
axes[1, 1].plot(x_arms, [s_crc[f"sl_ratio_{a}"] for a in CHROMOSOME_ARMS], '^-', color='#d62728', label="Kolorektal Kanser (CRC)", lw=1.5, markersize=4)
axes[1, 1].axhline(0.18, color='black', linestyle=':', label="Sağlıklı Referans (0.18)")
axes[1, 1].set_xticks(x_arms[::2])
axes[1, 1].set_xticklabels([CHROMOSOME_ARMS[i] for i in range(0, len(CHROMOSOME_ARMS), 2)], rotation=45, fontsize=8)
axes[1, 1].set_title("Genom Boyu cfDNA Fragmantasyon Profili (DELFI)", fontsize=12, fontweight='bold')
axes[1, 1].legend(loc="upper right", fontsize=8)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Klinik Likit Biyopsi Vaka Testi (Inference)

In [ ]:
def test_plasma_sample(vaka_adi, age, conc, modal, motif, base_sl, hotspots):
    sample = {
        "sample_id": "TEST", "patient_age": age, "cfdna_ng_ml": conc,
        "modal_fragment_bp": modal, "motif_CCCA_freq": motif
    }
    for arm in CHROMOSOME_ARMS:
        val = np.random.normal(base_sl, 0.015) + (0.11 if arm in hotspots else 0)
        sample[f"sl_ratio_{arm}"] = val
        
    df_s = extract_fragmentomic_features(pd.DataFrame([sample]))
    vec = scaler.transform(np.array([df_s[c].values[0] for c in feature_cols]).reshape(1, -1))
    probs = best_model.predict_proba(vec)[0]
    pred = int(np.argmax(probs))
    
    print(f"● {vaka_adi}")
    print(f"  Yapay Zeka Kararı : >>> {CLASS_NAMES[pred]} <<< (Güven: %{probs[pred]*100:.2f})")
    print(f"  Olasılıklar       : Sağlıklı: %{probs[0]*100:.1f} | Akciğer: %{probs[1]*100:.1f} | Kolorektal: %{probs[2]*100:.1f}\n")

test_plasma_sample("Vaka 1: 48 Yaşında Sağlıklı Birey", 48, 7.8, 167, 0.025, 0.175, [])
test_plasma_sample("Vaka 2: Evre I Akciğer Kanseri Şüphesi", 64, 14.9, 162, 0.016, 0.24, ["1q", "3q", "5p", "19q"])
test_plasma_sample("Vaka 3: Evre I Kolorektal Kanser Şüphesi", 59, 17.2, 161, 0.015, 0.25, ["13q", "18q", "20p", "20q"])